In [1]:
"""
causal_shap_alias.py  —  Causal SHAP + ALIAS  |  6 Datasets
=============================================================
ALIAS: DAG Learning with Efficient Unconstrained Policies
Duong, Le, Nguyen — arXiv:2408.13448

ALIAS is implemented directly from the paper (Algorithm 1 + Figure 4):
  Vec2DAG(z):
    p = z[:d]
    E[triu_indices(d,1)] = z[d:]
    A[i,j] = 1  iff  (E+E.T)[i,j]>0  AND  p[i]<p[j]   →  edge i→j

  Policy: π_θ = N(μ_θ, diag(σ²_θ)),  dim = d*(d+1)/2
  Training: REINFORCE with baseline (vanilla policy gradient)
    r^k = (1/n/d) * BIC(D, Vec2DAG(z^k))
    θ ← θ + η * (1/B) Σ ∇_θ log π_θ(z^k) * (r^k - baseline)

  Defaults from Table 6: batch=64, steps=20000, lr=3e-4, BIC-EV for linear,
                         BIC-NV for nonlinear/classification.

Two-gamma architecture (same as DrBO experiment):
  γ_est : OLS-estimated causal effect → used inside Causal SHAP + graph edges
  γ_gt  : SEM ground truth            → used only for evaluation metrics

COMPAS: drop perfectly collinear _0 dummy columns (condition number 1e18→51).

Setup:
  pip install scikit-learn xgboost pandas numpy scipy networkx matplotlib

Usage:
  python causal_shap_alias.py [--data_dir /path/to/datasets]
"""

import os, sys, time, warnings, argparse
import numpy as np
import pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx
from scipy.stats import pearsonr, spearmanr
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge

warnings.filterwarnings("ignore")

parser = argparse.ArgumentParser()
parser.add_argument("--data_dir",       default="../../../datasets")
parser.add_argument("--alias_steps",    type=int,   default=20000)
parser.add_argument("--alias_batch",    type=int,   default=64)
parser.add_argument("--alias_lr",       type=float, default=3e-4)
parser.add_argument("--n_shap_samples", type=int,   default=200)
parser.add_argument("--M",              type=int,   default=512)
parser.add_argument("--seed",           type=int,   default=42)
# MC Shapley params (Wine / COMPAS)
parser.add_argument("--mc_k",          type=int,   default=40)
parser.add_argument("--mc_bg_size",    type=int,   default=120)
parser.add_argument("--mc_iters",      type=int,   default=600)
parser.add_argument("--robust_k",      type=int,   default=15)
parser.add_argument("--perturb_runs",  type=int,   default=5)
parser.add_argument("--perturb_sigma", type=float, default=0.05)
args, _ = parser.parse_known_args()

DATA_DIR     = args.data_dir
OUT_DIR      = "results_alias"
BARCHART_DIR = os.path.join(OUT_DIR, "bar_charts")
GRAPH_DIR    = os.path.join(OUT_DIR, "graphs")
for _d in [OUT_DIR, BARCHART_DIR, GRAPH_DIR]:
    os.makedirs(_d, exist_ok=True)

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False


# Alias: vec2dag + bic scores + reinforce

def vec2dag(z: np.ndarray, d: int) -> np.ndarray:
    """
    Paper Fig 4. Map real vector z ∈ R^{d(d+1)/2} → binary DAG adjacency.
    A[i,j]=1  iff  (E+E.T)[i,j]>0  AND  p[i]<p[j]  →  edge i→j
    Acyclicity guaranteed by the node-potential ordering (Theorem 1).
    """
    p = z[:d]
    E = np.zeros((d, d))
    E[np.triu_indices(d, 1)] = z[d:]
    return (((E + E.T) > 0) * (p[:, None] < p[None, :])).astype(np.int8)


def _ssr(X_node: np.ndarray, X_parents: np.ndarray) -> float:
    """Sum of squared residuals of OLS regression."""
    if X_parents.shape[1] == 0:
        return float(np.sum(X_node**2))
    coef, *_ = np.linalg.lstsq(X_parents, X_node, rcond=None)
    return float(np.sum((X_node - X_parents @ coef)**2))


def bic_ev(X: np.ndarray, A: np.ndarray) -> float:
    """BIC equal-variance (higher = better). Paper eq for linear data."""
    n, d = X.shape
    total = sum(_ssr(X[:, j], X[:, np.where(A[:, j] == 1)[0]]) for j in range(d))
    return -n * d * np.log(total / (n * d) + 1e-12) - A.sum() * np.log(n)


def bic_nv(X: np.ndarray, A: np.ndarray) -> float:
    """BIC non-equal variance (higher = better). For classification/regression."""
    n, d = X.shape
    score = 0.0
    for j in range(d):
        ssr = _ssr(X[:, j], X[:, np.where(A[:, j] == 1)[0]])
        score += -n * np.log(ssr / n + 1e-12)
    return score - A.sum() * np.log(n)


def prune_linear(X: np.ndarray, A: np.ndarray, threshold: float = 0.3) -> np.ndarray:
    """
    Post-processing: threshold OLS edge weights.
    Paper Sec 4.2 + Table 6: threshold = 0.3 for linear data.
    """
    d = X.shape[1]
    A_pruned = A.copy()
    for j in range(d):
        parents = np.where(A[:, j] == 1)[0]
        if len(parents) == 0:
            continue
        coef, *_ = np.linalg.lstsq(X[:, parents], X[:, j], rcond=None)
        for ki, p in enumerate(parents):
            if abs(coef[ki]) < threshold:
                A_pruned[p, j] = 0
    return A_pruned


def run_alias(X_sc: np.ndarray, task: str, dname: str,
              n_steps: int, batch: int, lr: float, seed: int,
              prune: bool = True) -> tuple:
    """
    ALIAS causal discovery (Algorithm 1 of the paper).
    Returns (adj_binary, adj_weighted, elapsed_sec).

    adj_binary  : (d×d) int  — DAG adjacency matrix
    adj_weighted: (d×d) float — OLS edge weights on the recovered skeleton
    """
    rng = np.random.default_rng(seed)
    n, d = X_sc.shape
    dim  = d * (d + 1) // 2

    # Choose BIC: EV for regression-like, NV for classification
    score_fn = bic_ev if task == "regression" else bic_nv

    # Normalise reward scale for numerical stability (paper: divide by n*d)
    reward_scale = n * d

    # ── Policy initialisation ──────────────────────────────────────────────
    mu    = rng.standard_normal(dim) * 0.1
    log_s = np.full(dim, -1.0)    # σ ≈ 0.37 initially

    best_score = -np.inf
    best_dag   = np.zeros((d, d), dtype=np.int8)

    t0 = time.time()
    print(f"    Running ALIAS ({n_steps} steps, batch={batch}) …")

    for step in range(n_steps):
        sigma = np.exp(log_s)
        eps   = rng.standard_normal((batch, dim))
        zs    = mu + sigma * eps

        # Score each candidate DAG
        rewards = np.array([score_fn(X_sc, vec2dag(zs[k], d)) / reward_scale
                            for k in range(batch)])

        # REINFORCE with variance-reduction baseline (mean reward)
        baseline = rewards.mean()
        adv      = rewards - baseline

        # ∇_μ log π = (z−μ)/σ² = eps/σ
        # ∇_{log σ} log π = (eps²−1)
        grad_mu = np.mean(adv[:, None] * eps / sigma[None, :], axis=0)
        grad_ls = np.mean(adv[:, None] * (eps**2 - 1),        axis=0)

        mu    += lr * grad_mu
        log_s += lr * grad_ls
        log_s  = np.clip(log_s, -5, 2)    # keep σ in [~0.007, ~7.4]

        # Track global best
        for k in range(batch):
            s = score_fn(X_sc, vec2dag(zs[k], d))
            if s > best_score:
                best_score = s
                best_dag   = vec2dag(zs[k], d).copy()

        if (step + 1) % 5000 == 0:
            elapsed = time.time() - t0
            print(f"      step {step+1}/{n_steps}  best_score={best_score:.2f}"
                  f"  edges={int(best_dag.sum())}  t={elapsed:.0f}s")

    elapsed = time.time() - t0

    # ── Post-processing: linear pruning (paper Sec 4.2) ───────────────────
    if prune and task in ("regression", "multiclass", "binary"):
        # Use threshold 0.3 for linear, skip for small d (lung cancer d=3)
        thresh = 0.3 if d >= 5 else 0.1
        best_dag = prune_linear(X_sc, best_dag, threshold=thresh)

    n_edges = int(best_dag.sum())

    # ── Recover OLS edge weights on pruned skeleton ────────────────────────
    adj_weighted = np.zeros((d, d), dtype=float)
    for j in range(d):
        parents = np.where(best_dag[:, j] == 1)[0]
        if len(parents) == 0:
            continue
        r = Ridge(alpha=1e-3).fit(X_sc[:, parents], X_sc[:, j])
        for ki, p in enumerate(parents):
            adj_weighted[p, j] = float(r.coef_[ki])

    print(f"  [ALIAS] {dname}: {n_edges} edges, {elapsed:.1f}s"
          f"  (best BIC={best_score:.2f})")
    return best_dag, adj_weighted, elapsed


#  GROUND-TRUTH GAMMA (γ_gt) - soil / yield / cv_risk / lung_cancer only

def gt_gamma_soil(gt_path):
    gt  = pd.read_csv(gt_path)
    net = np.array([np.abs(gt['OC_base']  - gt['OC_penalty']).mean(),
                    np.abs(gt['N_base']   - gt['N_penalty']).mean(),
                    np.abs(gt['P_base']   - gt['P_penalty']).mean(),
                    np.abs(gt['K_base']   - gt['K_penalty']).mean(),
                    gt['PH_penalty'].abs().mean(),
                    np.abs(gt['ZN_base']  - gt['ZN_penalty']).mean(),
                    gt['EC_penalty'].abs().mean()])
    return net / net.sum()


def gt_gamma_yield():
    g = np.array([0.296, 0.397, 0.220, 0.088])   # NPZ order: N,I,P,H
    return g / g.sum()


def gt_gamma_cv_risk():
    var_diet = (9.0**2)/12.0; var_sleep = 4.0
    var_bmi  = 0.16*var_diet + 0.25*var_sleep + 1.0
    var_cv   = 2.25*var_bmi  + 3.0
    g = np.array([0.0, 0.0, 0.0, 2.25*var_bmi/var_cv])
    return g / g.sum()


def gt_gamma_lung_cancer():
    var_s, var_st, var_eps = 2.0, 2.0, 3.0
    var_y = 4.0*var_s + 1.44*var_st + var_eps
    raw   = np.array([4.0*var_s/var_y, 1.44*var_st/var_y, 0.0])
    return raw / raw[:2].sum()


# Dataset loaders

def _npz(fname):
    d = np.load(os.path.join(DATA_DIR, fname), allow_pickle=True)
    return (d['X_train'], d['X_test'], d['y_train'], d['y_test'],
            [str(f) for f in d['feature_names']])


def load_soil():
    Xtr,Xte,ytr,yte,fn = _npz("soil_spi_preprocessed.npz")
    X=np.concatenate([Xtr,Xte]); y=np.concatenate([ytr,yte])
    gt_path = os.path.join(DATA_DIR,"ground_truth.csv")
    gamma   = gt_gamma_soil(gt_path) if os.path.exists(gt_path) else None
    return X,X,y,y, gamma, fn, "multiclass","soil","SPI Class"


def load_yield():
    Xtr,Xte,ytr,yte,fn = _npz("yield_preprocessed.npz")
    X=np.concatenate([Xtr,Xte]); y=np.concatenate([ytr,yte])
    return X,X,y,y, gt_gamma_yield(), fn, "multiclass","yield","Crop Yield"


def load_cv_risk():
    csv=os.path.join(DATA_DIR,"cv_risk.csv"); feat=['diet_score','sleep_duration','family_history','bmi']
    df=pd.read_csv(csv).drop(columns=[c for c in pd.read_csv(csv,nrows=0).columns if 'Unnamed' in c])
    X=df[feat].values.astype(float); y=df['cv_risk'].values.astype(float)
    return X,X,y,y, gt_gamma_cv_risk(), feat, "regression","cv_risk","CV Risk"


def load_lung_cancer():
    csv=os.path.join(DATA_DIR,"lung_cancer_risk.csv"); feat=['smoking','stress','drink_coffee']
    df=pd.read_csv(csv).drop(columns=[c for c in pd.read_csv(csv,nrows=0).columns if 'Unnamed' in c])
    X=df[feat].values.astype(float); y=df['lung_cancer_risk'].values.astype(float)
    return X,X,y,y, gt_gamma_lung_cancer(), feat, "regression","lung_cancer","Lung Cancer Risk"


def load_wine():
    Xtr,Xte,ytr,yte,fn = _npz("wine_preprocessed.npz")
    return Xtr,Xte,ytr,yte, None, fn, "multiclass","wine","Wine Class"


def load_compas():
    Xtr,Xte,ytr,yte,fn = _npz("compas_preprocessed.npz")
    preferred=["age","juv_fel_count","juv_misd_count","juv_other_count","priors_count",
               "c_charge_degree_F","sex_1","age_cat_25-45_1","age_cat_Greaterthan45_1",
               "race_African-American_1"]
    kept=[c for c in preferred if c in fn]; idx=[fn.index(c) for c in kept]
    return Xtr[:,idx],Xte[:,idx],ytr,yte, None, kept, "binary","compas","Recidivism"


LOADERS = [load_soil, load_yield, load_cv_risk,
           load_lung_cancer, load_wine, load_compas]


#  ESTIMATED GAMMA (γ_est) - OLS on [features -> target]

def estimate_gamma(X, y, task):
    sc=StandardScaler(); Xs=sc.fit_transform(X)
    if task=="regression":
        coefs=np.abs(LinearRegression().fit(Xs,y).coef_)
    elif task=="binary":
        coefs=np.abs(LogisticRegression(max_iter=1000,C=1.0).fit(Xs,y).coef_[0])
    else:
        coefs=np.abs(LogisticRegression(max_iter=1000,C=1.0).fit(Xs,y).coef_).mean(0)
    coefs=np.maximum(coefs,0.0); return coefs/(coefs.sum()+1e-12)


#  CAUSAL SHAP  (γ_est-weighted permutation, causal ordering from ALIAS DAG)

def _causal_perm(adj, rng):
    d=adj.shape[0]; G=nx.DiGraph(); G.add_nodes_from(range(d))
    for i in range(d):
        for j in range(d):
            if adj[i,j]: G.add_edge(i,j)
    perm=[]
    for layer in nx.topological_generations(G):
        lst=list(layer); rng.shuffle(lst); perm.extend(lst)
    return perm


def _build_predictor(X, y, task):
    sc=StandardScaler(); Xs=sc.fit_transform(X)
    if task=="regression":
        m=LinearRegression().fit(Xs,y); return lambda Z: m.predict(sc.transform(Z))
    elif task=="binary":
        m=LogisticRegression(max_iter=1000,C=1.0).fit(Xs,y)
        return lambda Z: m.predict_proba(sc.transform(Z))[:,1]
    else:
        m=LogisticRegression(max_iter=1000,C=1.0).fit(Xs,y)
        return lambda Z: m.predict_proba(sc.transform(Z)).max(axis=1)


def _causal_shap_full(X_train, y_train, X_eval, adj, gamma_est, task):
    rng=np.random.default_rng(args.seed); n_tr,d=X_train.shape
    predict=_build_predictor(X_train,y_train,task)
    n_eval=min(args.n_shap_samples,len(X_eval))
    idx=rng.choice(len(X_eval),size=n_eval,replace=False)
    phi=np.zeros((n_eval,d))
    for si,x_ev in enumerate(X_eval[idx]):
        acc=np.zeros(d)
        for _ in range(args.M):
            perm=_causal_perm(adj,rng)
            x_curr=X_train[rng.integers(n_tr)].copy()
            prev=predict(x_curr.reshape(1,-1))[0]
            for f in perm:
                x_curr[f]=x_ev[f]; nxt=predict(x_curr.reshape(1,-1))[0]
                acc[f]+=gamma_est[f]*(nxt-prev); prev=nxt
        phi[si]=acc/args.M
    return phi,idx


#  MC SHAPLEY (reference for Wine / COMPAS)

def _build_classifier(X_train, y_train, task):
    if task=="binary" and HAS_XGB:
        m=XGBClassifier(n_estimators=250,max_depth=4,learning_rate=0.05,
                        subsample=0.85,colsample_bytree=0.85,random_state=args.seed,
                        verbosity=0,n_jobs=4,tree_method="hist",objective="binary:logistic")
        m.fit(X_train,y_train.astype(int)); classes=m.classes_
        pp=lambda Z: m.predict_proba(Z)
        pc=lambda Z: np.array([int(np.where(classes==r)[0][0]) for r in m.predict(Z)])
    else:
        sc=StandardScaler(); Xs=sc.fit_transform(X_train)
        m=LogisticRegression(max_iter=1000,C=1.0).fit(Xs,y_train); classes=m.classes_
        pp=lambda Z: m.predict_proba(sc.transform(Z))
        pc=lambda Z: np.array([int(np.where(classes==r)[0][0]) for r in m.predict(sc.transform(Z))])
    return pp,pc


def mc_shapley_global(X_train, y_train, X_test, task):
    rng=np.random.default_rng(args.seed); d=X_train.shape[1]
    pp,pc=_build_classifier(X_train,y_train,task)
    K=min(args.mc_k,len(X_test)); sel=rng.choice(len(X_test),size=K,replace=False)
    local=np.zeros((K,d))
    for cnt,i in enumerate(sel):
        if (cnt+1)%10==0: print(f"      MC Shapley {cnt+1}/{K} …",flush=True)
        x=X_test[i].astype(float); c=int(pc(x.reshape(1,-1))[0])
        bg=X_train[rng.choice(len(X_train),size=min(args.mc_bg_size,len(X_train)),replace=False)].copy()
        phi=np.zeros(d)
        for _ in range(args.mc_iters):
            perm=rng.permutation(d); curr=[]
            prev_v=float(np.mean(pp(bg)[:,c]))
            for p in perm:
                bg_mod=bg.copy()
                for fi in curr+[int(p)]: bg_mod[:,fi]=x[fi]
                new_v=float(np.mean(pp(bg_mod)[:,c]))
                phi[p]+=new_v-prev_v; curr=curr+[int(p)]; prev_v=new_v
        local[cnt]=phi/args.mc_iters
    ma=np.mean(np.abs(local),axis=0); return ma/(ma.sum()+1e-12),local,sel


# Metrics

def _safe_sp(a,b):
    a,b=np.asarray(a,float),np.asarray(b,float)
    if np.std(a)<1e-12 or np.std(b)<1e-12: return np.nan
    r,_=spearmanr(a,b); return float(r)

def _safe_pr(a,b):
    a,b=np.asarray(a,float),np.asarray(b,float)
    if np.std(a)<1e-12 or np.std(b)<1e-12: return np.nan
    r,_=pearsonr(a,b); return float(r)

def _cos(a,b):
    a,b=np.asarray(a,float),np.asarray(b,float)
    dn=np.linalg.norm(a)*np.linalg.norm(b)
    return float(np.dot(a,b)/dn) if dn>=1e-12 else np.nan

def _norm(v):
    v=np.abs(np.asarray(v,float)); s=v.sum()
    return v/s if s>1e-12 else np.ones_like(v)/len(v)


def gt_metrics(mean_shap, gamma_gt):
    a=_norm(mean_shap); g=_norm(gamma_gt)
    return dict(pearson=round(_safe_pr(a,g),4), spearman=round(_safe_sp(a,g),4),
                cosine_sim=round(_cos(a,g),4),
                rmse=round(float(np.sqrt(np.mean((a-g)**2))),6))


def faithfulness_metrics(X_test, phi, eval_idx, pp, pc, feat_means, d):
    sp,pr,cs=[],[],[]
    for ri in range(len(eval_idx)):
        x=X_test[eval_idx[ri]].astype(float); c=int(pc(x.reshape(1,-1))[0])
        base=float(pp(x.reshape(1,-1))[0,c])
        drops=[]
        for j in range(d):
            xm=x.copy(); xm[j]=feat_means[j]
            drops.append(base-float(pp(xm.reshape(1,-1))[0,c]))
        attr=np.abs(phi[ri]); drops=np.abs(np.array(drops,float))
        r=_safe_sp(attr,drops);
        if not np.isnan(r): sp.append(r)
        r=_safe_pr(attr,drops);
        if not np.isnan(r): pr.append(r)
        r=_cos(attr,drops);
        if not np.isnan(r): cs.append(r)
    return (np.mean(sp) if sp else np.nan,
            np.mean(pr) if pr else np.nan,
            np.mean(cs) if cs else np.nan)


def robustness_metrics(X_train,X_test,y_train,adj,gamma_est,task,feat_stds,pp,pc):
    rng=np.random.default_rng(args.seed); K=min(args.robust_k,len(X_test))
    sel=rng.choice(len(X_test),size=K,replace=False)
    cos_l,rank_l,l2_l=[],[],[]
    for k,i in enumerate(sel):
        x0=X_test[i].astype(float)
        phi0,_=_causal_shap_full(X_train,y_train,x0.reshape(1,-1),adj,gamma_est,task)
        phi0=np.abs(phi0[0])
        for r in range(args.perturb_runs):
            xp=(x0+rng.normal(0,args.perturb_sigma,len(feat_stds))*feat_stds).reshape(1,-1)
            phip,_=_causal_shap_full(X_train,y_train,xp,adj,gamma_est,task)
            phip=np.abs(phip[0])
            cos_l.append(_cos(phi0,phip))
            rk=_safe_sp(phi0,phip)
            if not np.isnan(rk): rank_l.append(rk)
            l2_l.append(float(np.linalg.norm(phi0-phip)))
    return (np.mean(cos_l)  if cos_l  else np.nan,
            np.mean(rank_l) if rank_l else np.nan,
            np.mean(l2_l)   if l2_l   else np.nan)


#  GRAPH VISUALISATION  (same aesthetic as DrBO script)

POS_COL='#1B8C7A'; NEG_COL='#C0392B'; EST_COL='#8E24AA'
TARGET_COL='#6A1B9A'; BLUES=['#AED6F1','#5DADE2','#2E86C1','#1B5E87','#0D3349']

def _wrap(name,ml=12):
    p=name.split()
    if len(name)<=ml or len(p)==1: return name
    mid=len(p)//2; return ' '.join(p[:mid])+'\n'+' '.join(p[mid:])


def draw_causal_graph(adj_w, feat_names, gamma_est, target_name, title, out_path):
    d=len(feat_names)
    G=nx.DiGraph(); G.add_nodes_from(range(d))
    for i in range(d):
        for j in range(d):
            if abs(adj_w[i,j])>1e-6: G.add_edge(i,j,weight=float(adj_w[i,j]))
    try: layers=list(nx.topological_generations(G))
    except: layers=[list(range(d))]
    pos={}; nl=len(layers)
    for li,layer in enumerate(layers):
        layer=sorted(layer); n=len(layer)
        y=1.0-li*(0.80/max(nl-1,1))
        for ki,node in enumerate(layer): pos[node]=((ki+1)/(n+1),y)
    TARGET=d; pos[TARGET]=(0.5,-0.08)
    figw=max(12,d*1.3); figh=max(10,(nl+2)*1.8)
    fig,ax=plt.subplots(figsize=(figw,figh))
    fig.patch.set_facecolor('white'); ax.set_facecolor('white')
    # Inter-feature edges
    feat_labels=[]
    for src,dst,data in G.edges(data=True):
        w=data['weight']; col=POS_COL if w>0 else NEG_COL
        lw=0.9+min(abs(w)*2.5,4.0)
        xi,yi=pos[src]; xj,yj=pos[dst]
        rad=0.0 if abs(yi-yj)>0.35 else (0.25 if abs(yi-yj)<0.05 else 0.12)
        ax.annotate("",xy=(xj,yj),xytext=(xi,yi),
            arrowprops=dict(arrowstyle="-|>",color=col,lw=lw,alpha=0.72,
                            connectionstyle=f"arc3,rad={rad}",shrinkA=23,shrinkB=23))
        t=0.42; mx=xi+t*(xj-xi); my=yi+t*(yj-yi)
        dx,dy=xj-xi,yj-yi; norm=max(np.hypot(dx,dy),1e-6)
        side=1 if (src+dst)%2==0 else -1
        ox=side*(-dy/norm)*0.04; oy=side*(dx/norm)*0.04
        feat_labels.append((mx+ox,my+oy,f"{w:+.2f}",'white','#cccccc'))
    # Feature->target edges (γ_est, dashed purple)
    feat_sorted=sorted(range(d),key=lambda i:pos[i][0])
    tx,ty=pos[TARGET]; tgt_labels=[]
    for rank,fi in enumerate(feat_sorted):
        w=float(gamma_est[fi])
        if w<1e-6: continue
        xi,yi=pos[fi]; lw=0.8+min(w*30,3.5)
        spread=min(0.28,d*0.04); arr_x=tx+spread*(rank/(max(d-1,1))-0.5)
        ax.annotate("",xy=(arr_x,ty+0.065),xytext=(xi,yi),
            arrowprops=dict(arrowstyle="-|>",color=EST_COL,lw=lw,alpha=0.60,
                            linestyle='dashed',connectionstyle="arc3,rad=0.0",
                            shrinkA=23,shrinkB=26))
        t=0.55; mx=xi+t*(arr_x-xi); my=yi+t*(ty+0.065-yi)
        dx,dy=arr_x-xi,ty+0.065-yi; norm=max(np.hypot(dx,dy),1e-6)
        side=1 if fi%2==0 else -1
        tgt_labels.append((mx+side*(-dy/norm)*0.035, my+side*(dx/norm)*0.035, f"{w:.3f}"))
    for lx,ly,lab,fc,ec in feat_labels:
        ax.text(lx,ly,lab,ha='center',va='center',fontsize=7.5,color='#111111',
                fontweight='medium',zorder=8,
                bbox=dict(boxstyle='round,pad=0.18',fc=fc,ec=ec,alpha=0.90,lw=0.5))
    for lx,ly,lab in tgt_labels:
        ax.text(lx,ly,lab,ha='center',va='center',fontsize=7.5,color='#4A148C',
                fontweight='semibold',zorder=10,
                bbox=dict(boxstyle='round,pad=0.15',fc='#F3E5F5',ec='#CE93D8',alpha=0.93,lw=0.5))
    in_deg=dict(G.in_degree())
    for i in range(d):
        x,y=pos[i]; col=BLUES[min(in_deg.get(i,0),4)]
        label=_wrap(feat_names[i]); lines=label.count('\n')+1
        bw=max(0.10,len(max(label.split('\n'),key=len))*0.013); bh=0.055+(lines-1)*0.033
        ax.add_patch(mpatches.FancyBboxPatch((x-bw,y-bh/2),bw*2,bh,
            boxstyle="round,pad=0.015",facecolor=col,edgecolor='white',linewidth=2,zorder=5))
        ax.text(x,y,label,ha='center',va='center',fontsize=8.5,color='white',
                fontweight='bold',zorder=6,multialignment='center',linespacing=1.25)
    tbw=max(0.14,len(target_name)*0.013)
    ax.add_patch(mpatches.FancyBboxPatch((tx-tbw,ty-0.065),tbw*2,0.13,
        boxstyle="round,pad=0.018",facecolor=TARGET_COL,edgecolor='white',linewidth=2.5,zorder=5))
    ax.text(tx,ty,target_name,ha='center',va='center',fontsize=10,color='white',fontweight='bold',zorder=6)
    handles=[
        plt.Line2D([0],[0],color=POS_COL,lw=2,label='Positive inter-feature effect (ALIAS)'),
        plt.Line2D([0],[0],color=NEG_COL,lw=2,label='Negative inter-feature effect (ALIAS)'),
        plt.Line2D([0],[0],color=EST_COL,lw=2,ls='--',label='Feature → target (γ_est, OLS)'),
        mpatches.Patch(facecolor=BLUES[3],label='High in-degree feature'),
        mpatches.Patch(facecolor=BLUES[0],label='Low in-degree feature'),
        mpatches.Patch(facecolor=TARGET_COL,label='Target variable'),
    ]
    ax.legend(handles=handles,loc='lower left',fontsize=8.5,framealpha=0.95,
              edgecolor='#cccccc',fancybox=True,ncol=2)
    ax.set_title(title,fontsize=14,fontweight='bold',pad=16,color='#111111')
    ax.set_xlim(-0.05,1.05); ax.set_ylim(-0.28,1.25); ax.axis('off')
    fig.tight_layout(pad=2.2)
    fig.savefig(out_path,dpi=300,bbox_inches='tight',facecolor='white')
    plt.close(); print(f"    graph img  → {out_path}")


def save_graph_files(adj_w, feat_names, gamma_est, target_name, dname):
    tag=f"{dname}_alias"; d=len(feat_names)
    pd.DataFrame(adj_w,index=feat_names,columns=feat_names).to_csv(
        os.path.join(OUT_DIR,f"{tag}_adj_matrix.csv"))
    rows=[]
    for i in range(d):
        for j in range(d):
            if abs(adj_w[i,j])>1e-6:
                rows.append({"source":feat_names[i],"target_node":feat_names[j],
                             "weight":round(adj_w[i,j],6),"edge_type":"inter_feature"})
    for i,f in enumerate(feat_names):
        rows.append({"source":f,"target_node":target_name,
                     "weight":round(float(gamma_est[i]),6),
                     "edge_type":"feature_to_target_gamma_est"})
    pd.DataFrame(rows).to_csv(os.path.join(OUT_DIR,f"{tag}_graph_edges.csv"),index=False)
    draw_causal_graph(adj_w,feat_names,gamma_est,target_name,
                      f"ALIAS Causal Graph — {target_name}",
                      os.path.join(GRAPH_DIR,f"{tag}_graph.png"))


# Bar chart

def plot_bar(mean_shap, feat_names, dname, gamma_gt=None, mc_ref=None):
    cmap=plt.get_cmap("tab10"); colors=[cmap(i%10) for i in range(len(feat_names))]
    fig,ax=plt.subplots(figsize=(max(10,len(feat_names)*0.8),5))
    ax.bar(feat_names,mean_shap,color=colors,edgecolor='none')
    if gamma_gt is not None and len(gamma_gt)==len(feat_names):
        gn=gamma_gt/(gamma_gt.sum()+1e-12)*mean_shap.sum()
        ax.plot(feat_names,gn,color='black',lw=1.8,ls='--',marker='o',ms=5,
                label='Ground Truth γ (SEM)',zorder=5)
    if mc_ref is not None:
        mc_n=_norm(mc_ref)*mean_shap.sum()
        ax.plot(feat_names,mc_n,color='crimson',lw=1.8,ls=':',marker='s',ms=5,
                label='MC Shapley reference',zorder=5)
    if gamma_gt is not None or mc_ref is not None: ax.legend(fontsize=10)
    ax.set_xlabel("Feature",fontsize=11); ax.set_ylabel("Mean |Causal SHAP|",fontsize=11)
    ax.set_title(f"Causal SHAP (ALIAS) — {dname}",fontsize=12,fontweight='bold')
    ax.set_xticks(range(len(feat_names)))
    ax.set_xticklabels(feat_names,rotation=90,ha='center',fontsize=8)
    ax.spines[['top','right']].set_visible(False)
    fig.tight_layout()
    out=os.path.join(BARCHART_DIR,f"{dname}_alias_causal_shap.png")
    fig.savefig(out,dpi=600,bbox_inches='tight'); plt.close()
    print(f"    bar chart  → {out}")


# Per-dataset pipeline

def run_dataset(loader):
    try: out=loader()
    except FileNotFoundError as e: print(f"  [SKIP] {e}"); return None
    X_train,X_test,y_train,y_test,gamma_gt,feat_names,task,dname,target_name=out
    n,d=X_train.shape; has_gt=gamma_gt is not None and len(gamma_gt)==d
    print(f"\n{'='*60}")
    print(f"  {dname.upper()}  n_train={n}  d={d}  task={task}")
    if has_gt:
        print(f"  γ_gt: { {feat_names[i]:round(float(gamma_gt[i]),4) for i in range(d)} }")
    else:
        print(f"  γ_gt: None → metrics vs MC Shapley + faithfulness + robustness")
    print(f"{'='*60}")
    X_sc=StandardScaler().fit_transform(X_train)

    # 1. ALIAS causal discovery
    print("  [1/5] Causal discovery (ALIAS) …")
    adj_bin,adj_w,t_disc=run_alias(X_sc,task,dname,
                                   args.alias_steps,args.alias_batch,args.alias_lr,args.seed)

    # 2. γ_est via OLS
    print("  [2/5] Estimating γ_est via OLS …")
    gamma_est=estimate_gamma(X_train,y_train,task)
    print(f"    γ_est: { {feat_names[i]:round(float(gamma_est[i]),4) for i in range(d)} }")

    # 3. Causal SHAP
    print("  [3/5] Causal SHAP (γ_est-weighted) …")
    t1=time.time()
    phi,eval_idx=_causal_shap_full(X_train,y_train,X_test,adj_bin,gamma_est,task)
    t_shap=time.time()-t1; mean_shap=np.mean(np.abs(phi),axis=0)
    shap_df=pd.DataFrame(phi,columns=feat_names); shap_df.insert(0,"sample_idx",eval_idx)
    shap_df.to_csv(os.path.join(OUT_DIR,f"{dname}_alias_causal_shap_values.csv"),index=False)
    print(f"    done {t_shap:.1f}s")

    # 4. Graph
    print("  [4/5] Saving graph …")
    save_graph_files(adj_w,feat_names,gamma_est,target_name,dname)

    # 5. Metrics & charts
    print("  [5/5] Metrics & charts …")
    row=dict(dataset=dname,n_train=n,n_features=d,task=task,target=target_name,
             n_inter_edges=int(adj_bin.sum()),
             time_disc_s=round(t_disc,2),time_shap_s=round(t_shap,2))
    mc_ref_attr=None

    if has_gt:
        m=gt_metrics(mean_shap,gamma_gt); row.update(m)
        row.update(faith_spearman=np.nan,faith_pearson=np.nan,faith_cosine=np.nan,
                   pred_align_mc=np.nan,attr_cosine_mc=np.nan,
                   robustness_cosine=np.nan,rank_robustness=np.nan,l2=np.nan)
        print(f"    SHAP vs γ_gt: {m}")
        plot_bar(mean_shap,feat_names,dname,gamma_gt=gamma_gt)
    else:
        row.update(pearson=np.nan,spearman=np.nan,cosine_sim=np.nan,rmse=np.nan)
        print("    Computing MC Shapley reference …")
        mc_ref_attr,mc_local,mc_sel=mc_shapley_global(X_train,y_train,X_test,task)
        pp,pc=_build_classifier(X_train,y_train,task)
        feat_means=X_train.mean(axis=0)
        fsp,fpr,fcs=faithfulness_metrics(X_test,phi,eval_idx,pp,pc,feat_means,d)
        an=_norm(mean_shap); ref=_norm(mc_ref_attr)
        pred_align=_safe_pr(an,ref); attr_cos=_cos(an,ref)
        feat_stds=X_train.std(axis=0)+1e-8
        rob_cos,rank_rob,l2=robustness_metrics(X_train,X_test,y_train,adj_bin,
                                               gamma_est,task,feat_stds,pp,pc)
        row.update(faith_spearman=round(fsp,4) if not np.isnan(fsp) else np.nan,
                   faith_pearson =round(fpr,4) if not np.isnan(fpr) else np.nan,
                   faith_cosine  =round(fcs,4) if not np.isnan(fcs) else np.nan,
                   pred_align_mc =round(pred_align,4) if not np.isnan(pred_align) else np.nan,
                   attr_cosine_mc=round(attr_cos,4),
                   robustness_cosine=round(rob_cos,4) if not np.isnan(rob_cos) else np.nan,
                   rank_robustness  =round(rank_rob,4) if not np.isnan(rank_rob) else np.nan,
                   l2=round(l2,4) if not np.isnan(l2) else np.nan)
        print(f"    Faith: Sp={fsp:.4f}  Pr={fpr:.4f}  Cos={fcs:.4f}")
        print(f"    vs MC: align={pred_align:.4f}  cosine={attr_cos:.4f}")
        print(f"    Robustness: cos={rob_cos:.4f}  rank={rank_rob:.4f}  L2={l2:.4f}")
        plot_bar(mean_shap,feat_names,dname,mc_ref=mc_ref_attr)

    pd.DataFrame([row]).to_csv(os.path.join(OUT_DIR,f"{dname}_alias_metrics.csv"),index=False)
    imp=pd.DataFrame({"feature":feat_names,"mean_abs_shap":mean_shap,
                      "gamma_est":gamma_est,
                      "gamma_gt":gamma_gt if has_gt else [np.nan]*d})
    imp=imp.sort_values("mean_abs_shap",ascending=False).reset_index(drop=True)
    imp.to_csv(os.path.join(OUT_DIR,f"{dname}_alias_feature_importance.csv"),index=False)
    print(f"\n  Feature importance ({dname}):\n{imp.to_string(index=False)}")
    return row


# Summary plots

def plot_summary_gt(df):
    ms=["pearson","spearman","cosine_sim","rmse"]; dfp=df.dropna(subset=ms)
    if dfp.empty: return
    cmap=plt.get_cmap("tab10")
    fig,axes=plt.subplots(1,4,figsize=(22,5))
    for ax,m in zip(axes,ms):
        v=dfp[m].values; labels=dfp["dataset"].values
        ax.bar(labels,v,color=[cmap(i%10) for i in range(len(labels))],edgecolor='none')
        ax.set_title(m,fontsize=12,fontweight='bold')
        ax.set_xticklabels(labels,rotation=90,fontsize=9)
        ax.spines[['top','right']].set_visible(False)
        if m!="rmse": ax.set_ylim(-1,1)
    fig.suptitle("Causal SHAP (ALIAS) vs Ground Truth γ_gt",fontsize=13,fontweight='bold')
    fig.tight_layout()
    out=os.path.join(BARCHART_DIR,"summary_gt_metrics.png")
    fig.savefig(out,dpi=600,bbox_inches='tight'); plt.close()
    print(f"  GT summary → {out}")


def plot_summary_mc(df):
    ms=["faith_spearman","faith_pearson","faith_cosine",
        "pred_align_mc","attr_cosine_mc","robustness_cosine","rank_robustness","l2"]
    dfp=df.dropna(subset=["faith_spearman"])
    if dfp.empty: return
    cmap=plt.get_cmap("tab10")
    fig,axes=plt.subplots(2,4,figsize=(22,10))
    for ax,m in zip(axes.flat,ms):
        v=dfp[m].values if m in dfp.columns else []
        labels=dfp["dataset"].values
        if len(v):
            ax.bar(labels,v,color=[cmap(i%10) for i in range(len(labels))],edgecolor='none')
        ax.set_title(m,fontsize=11,fontweight='bold')
        ax.set_xticklabels(labels,rotation=90,fontsize=9)
        ax.spines[['top','right']].set_visible(False)
    fig.suptitle("Causal SHAP (ALIAS) — Wine/COMPAS metrics vs MC Shapley",
                 fontsize=13,fontweight='bold')
    fig.tight_layout()
    out=os.path.join(BARCHART_DIR,"summary_mc_metrics.png")
    fig.savefig(out,dpi=600,bbox_inches='tight'); plt.close()
    print(f"  MC summary → {out}")


def main():
    print("\n"+"="*65)
    print("  Causal SHAP + ALIAS  |  6 Datasets")
    print(f"  data_dir={DATA_DIR}  alias_steps={args.alias_steps}")
    print(f"  n_shap_samples={args.n_shap_samples}  M={args.M}")
    print("="*65)
    rows=[r for loader in LOADERS if (r:=run_dataset(loader)) is not None]
    if rows:
        summary=pd.DataFrame(rows)
        summary.to_csv(os.path.join(OUT_DIR,"all_datasets_summary.csv"),index=False)
        print(f"\n{'='*65}\nSUMMARY\n{'='*65}")
        print(summary.to_string(index=False))
        plot_summary_gt(summary); plot_summary_mc(summary)
    print("\nAll done.")

if __name__=="__main__":
    main()


  Causal SHAP + ALIAS  |  6 Datasets
  data_dir=../datasets  alias_steps=20000
  n_shap_samples=200  M=512

  SOIL  n_train=5000  d=7  task=multiclass
  γ_gt: {'Organic Carbon': 0.0622, 'Nitrogen': 0.4012, 'Phosphorus': 0.0762, 'Potassium': 0.2588, 'pH': 0.0886, 'Zinc': 0.0147, 'Electrical Conductivity': 0.0984}
  [1/5] Causal discovery (ALIAS) …
    Running ALIAS (20000 steps, batch=64) …
      step 5000/20000  best_score=3784.44  edges=9  t=568s
      step 10000/20000  best_score=3788.60  edges=9  t=1141s
      step 15000/20000  best_score=3788.60  edges=9  t=1720s
      step 20000/20000  best_score=3788.60  edges=9  t=2303s
  [ALIAS] soil: 1 edges, 2302.8s  (best BIC=3788.60)
  [2/5] Estimating γ_est via OLS …
    γ_est: {'Organic Carbon': 0.0891, 'Nitrogen': 0.2632, 'Phosphorus': 0.0675, 'Potassium': 0.2684, 'pH': 0.1694, 'Zinc': 0.0249, 'Electrical Conductivity': 0.1175}
  [3/5] Causal SHAP (γ_est-weighted) …
    done 117.5s
  [4/5] Saving graph …
    graph img  → results_alias/g